# Análisis Exploratorio con el Dataset Gapminder

## ¿Qué es Gapminder?

[Gapminder](https://www.gapminder.org/) es una fundación sueca que recopila y divulga estadísticas sobre el desarrollo mundial. Su objetivo es combatir la visión distorsionada que tenemos del mundo mostrando datos reales sobre salud, riqueza y población.

Este cuaderno analiza el dataset **Gapminder** integrado en Plotly Express, con datos socioeconómicos de **142 países** recogidos cada 5 años entre **1952 y 2007**.

## Variables del dataset

| Variable | Tipo | Descripción |
|---------|------|-------------|
| `country` | Categórica | Nombre del país |
| `continent` | Categórica | Continente (5 categorías) |
| `year` | Numérica | Año (1952–2007, cada 5 años → 12 observaciones) |
| `lifeExp` | Numérica | Esperanza de vida al nacer (años) |
| `pop` | Numérica | Población total |
| `gdpPercap` | Numérica | PIB per cápita en USD constantes |
| `iso_alpha` | Categórica | Código ISO-3 del país (usado en mapas) |

## Objetivos del análisis

1. Explorar la estructura y calidad del dataset.
2. Visualizar la distribución geográfica de riqueza, población y longevidad.
3. Analizar la relación entre desarrollo económico y calidad de vida.
4. Comparar tendencias históricas entre continentes.

## Índice de visualizaciones

| Sección | Gráfico | Pregunta que responde |
|---------|---------|----------------------|
| 2. Carga | Exploración tabular | ¿Qué cubre el dataset? |
| 3. Coropléticos | `px.choropleth` | ¿Cómo se distribuyen las variables en el mapa? |
| 4. Dispersión | `px.scatter` (estático + animado) | ¿Existe relación PIB ↔ longevidad? ¿Cómo evolucionó? |
| 5. Barras | `px.bar` | ¿Qué continente tiene más población? |
| 6. Líneas | `px.line` | ¿Cómo han evolucionado los indicadores? |
| 7. Histograma | `px.histogram` | ¿Cómo se distribuye la esperanza de vida? |
| 8. Cajas | `px.box` | ¿Cómo varía la distribución intra-continente? |
| 9. Calor | `px.imshow` | ¿Qué correlaciones y patrones temporales existen? |

> **Fuente:** `plotly.express.data.gapminder()` — datos originales de la Gapminder Foundation.

In [ ]:
%pip install pandas plotly nbformat --quiet

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")

## 1. Importación de Librerías

| Librería | Propósito en este notebook |
|----------|---------------------------|
| `pandas` | Carga de CSV, inspección y filtrado de datos |
| `plotly.express` | Todas las visualizaciones interactivas |
| `numpy` | Transformaciones numéricas (ej. escala logarítmica para población) |

## 2. Carga y Exploración del Dataset

El dataset Gapminder está disponible directamente en Plotly Express mediante `px.data.gapminder()`. Lo guardamos también como CSV para mostrar el flujo de trabajo habitual con ficheros locales.

### ¿Qué esperar del dataset?
- **Formato largo**: cada fila es una observación (país, año).
- **Sin valores nulos**: dataset limpio y listo para visualizar.
- **Escala temporal**: 12 puntos (1952, 1957, 1962, …, 2007), uno cada 5 años.
- **Rango de PIB**: de ~241 USD (Congo, 2002) a ~49.357 USD (Kuwait, 1957).

In [ ]:
# Cargar el dataset de Gapminder desde Plotly Express
data = px.data.gapminder()

print(f"Dimensiones  : {data.shape[0]} filas x {data.shape[1]} columnas")
print(f"Paises       : {data['country'].nunique()}")
print(f"Continentes  : {data['continent'].nunique()} — {sorted(data['continent'].unique())}")
print(f"Anos         : {sorted(data['year'].unique())}")

# Guardar como CSV (flujo habitual en proyectos reales)
data.to_csv('gapminder_dataset.csv', index=False)
print("\nGuardado como 'gapminder_dataset.csv'")

In [ ]:
data = pd.read_csv('gapminder_dataset.csv')

# Estructura del dataset
print("=== Tipos de datos ===")
print(data.dtypes.to_string())

print("\n=== Valores nulos por columna ===")
nulos = data.isnull().sum()
print(nulos.to_string() if nulos.sum() > 0 else "Ninguno — dataset completo")

print("\n=== Estadisticas descriptivas ===")
print(data[['lifeExp', 'pop', 'gdpPercap']].describe().round(2).to_string())

print("\n=== Top 5 paises por PIB per capita en 2007 ===")
top5_gdp = data[data['year']==2007].nlargest(5, 'gdpPercap')[['country','continent','gdpPercap','lifeExp']]
print(top5_gdp.to_string(index=False))

# Subconjunto 2007 — reutilizado en todos los graficos estaticos
data_2007 = data[data['year'] == 2007].copy()
print(f"\nRegistros en 2007: {len(data_2007)} paises")

data.head()

## 3. Mapas Coropléticos

Un **mapa coroplético** colorea cada región geográfica según el valor de una variable, permitiendo identificar patrones globales de un vistazo. Usamos la proyección *Natural Earth* para una representación equilibrada de continentes.

### 3.1 PIB per cápita (2007)

El **PIB per cápita** es el indicador económico estándar para comparar el nivel de vida entre países (aunque no mide la desigualdad interna).

#### ¿Qué esperamos ver?
- **Norteamérica, Europa Occidental, Japón y Australia**: colores más oscuros (economías maduras).
- **África Subsahariana**: colores claros, con excepciones en países exportadores de petróleo.
- **Asia y Latinoamérica**: alta heterogeneidad — desde economías muy pobres hasta emergentes avanzadas.

> Pasa el cursor sobre cada país para ver su esperanza de vida y población además del PIB.

In [ ]:
fig = px.choropleth(
    data_2007,
    locations="iso_alpha",
    color="gdpPercap",
    hover_name="country",
    hover_data={"continent": True, "lifeExp": True, "pop": True, "iso_alpha": False},
    color_continuous_scale="brwnyl",
    labels={
        "gdpPercap": "PIB per capita (USD)",
        "lifeExp": "Esp. de vida (años)",
        "pop": "Poblacion",
        "continent": "Continente"
    },
    title="PIB per Capita por Pais (2007)",
)
fig.update_geos(showcoastlines=True, coastlinecolor="Black", projection_type="natural earth")
fig.update_layout(
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
    coloraxis_colorbar=dict(title="USD/hab")
)
fig.show()

### 3.2 Población Total (2007)

La **población total** determina el peso demográfico de un país, pero no su nivel de desarrollo. China e India juntas concentran más del 37% de la población mundial.

#### ¿Qué esperamos ver?
- **Asia** domina visualmente: China (~1.320 M) e India (~1.110 M) destacan sobre todos los demás.
- **Europa y Norteamérica** tienen poblaciones moderadas pese a su alto PIB per cápita.
- **África** muestra crecimiento acelerado, especialmente en el África Subsahariana.

> Se aplica una **escala logarítmica** (log₁₀) al color para evitar que China e India "secuestren" visualmente la paleta y permita apreciar las diferencias en el resto de países.

In [ ]:
# Escala logaritmica: evita que China e India dominen toda la paleta de color
data_2007_pop = data_2007.copy()
data_2007_pop['log_pop'] = np.log10(data_2007_pop['pop'])

fig = px.choropleth(
    data_2007_pop,
    locations="iso_alpha",
    color="log_pop",
    hover_name="country",
    hover_data={"continent": True, "pop": True, "log_pop": False, "iso_alpha": False},
    color_continuous_scale="deep",
    labels={"log_pop": "log10(Poblacion)", "continent": "Continente", "pop": "Poblacion"},
    title="Poblacion por Pais (2007) — Escala logaritmica",
)
fig.update_geos(showcoastlines=True, coastlinecolor="Black", projection_type="natural earth")
fig.update_layout(
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
    coloraxis_colorbar=dict(
        title="log10(hab)",
        tickvals=[6, 7, 8, 9],
        ticktext=["1 M", "10 M", "100 M", "1.000 M"]
    )
)
fig.show()

### 3.3 Esperanza de Vida (2007)

La **esperanza de vida al nacer** refleja la calidad del sistema sanitario, la nutrición, la seguridad y las condiciones generales de vida. Es uno de los componentes del Índice de Desarrollo Humano (IDH) de la ONU.

#### ¿Qué esperamos ver?
- **Europa Occidental y Japón**: valores superiores a 80 años.
- **África Subsahariana**: valores en muchos casos por debajo de 55 años, fuertemente afectados por el VIH/SIDA.
- **Asia y Latinoamérica**: amplio rango, con mejoras notables respecto a décadas anteriores.

> La paleta `rdylgn` (rojo → amarillo → verde) es intuitiva: rojo = baja esperanza de vida, verde = alta. Compara este mapa con el del PIB — la correlación es alta pero **imperfecta**: Cuba tiene esperanza de vida similar a EE.UU. con una fracción de su PIB.

In [ ]:
fig = px.choropleth(
    data_2007,
    locations="iso_alpha",
    color="lifeExp",
    hover_name="country",
    hover_data={"continent": True, "gdpPercap": True, "pop": True, "iso_alpha": False},
    color_continuous_scale="rdylgn",
    labels={
        "lifeExp": "Esperanza de vida (años)",
        "continent": "Continente",
        "gdpPercap": "PIB per capita (USD)",
        "pop": "Poblacion"
    },
    title="Esperanza de Vida por Pais (2007)",
)
fig.update_geos(showcoastlines=True, coastlinecolor="Black", projection_type="natural earth")
fig.update_layout(
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
    coloraxis_colorbar=dict(title="Años")
)
fig.show()

## 4. Gráfico de Dispersión: PIB per cápita vs. Esperanza de Vida

### 4.1 Vista estática (2007)

El **scatter plot** posiciona cada país como un punto según dos variables continuas. Aquí:
- **Eje X**: PIB per cápita en escala logarítmica (los datos van de ~300 a ~50.000 USD).
- **Eje Y**: esperanza de vida.
- **Color**: continente.
- **Tamaño de la burbuja**: población — países más grandes tienen burbujas más grandes.

#### La relación PIB ↔ longevidad no es lineal

A partir de ~10.000–15.000 USD per cápita, las mejoras en esperanza de vida se desaceleran. Los países muy ricos ganan pocos años adicionales por cada dólar extra invertido. Este efecto de **rendimientos decrecientes** es visible en la curva que forman los puntos.

### 4.2 Visualización animada: 55 años de historia (1952–2007)

La segunda visualización reproduce la **gráfica icónica de Hans Rosling** presentada en su charla TED *"The best stats you've ever seen"* (2006, +15 M vistas).

- El tamaño de la burbuja representa la **población** del país.
- La animación muestra cómo el mundo entero se desplaza hacia la esquina **superior-derecha** (más rico y más longevo) en 55 años.
- **Asia Oriental** protagoniza el ascenso más espectacular (China, Corea del Sur).
- **África** mejora, pero a un ritmo más lento y con retroceso en los años 90 por el VIH/SIDA.

> Pulsa ▶ para ver la animación o arrastra el slider del año manualmente.

In [ ]:
# Scatter estatico 2007: PIB per capita vs Esperanza de vida
fig = px.scatter(
    data_2007,
    x="gdpPercap",
    y="lifeExp",
    color="continent",
    size="pop",
    size_max=50,
    hover_name="country",
    hover_data={"pop": True, "gdpPercap": True, "iso_alpha": False},
    log_x=True,
    labels={
        "gdpPercap": "PIB per capita (USD, escala log)",
        "lifeExp": "Esperanza de vida (años)",
        "pop": "Poblacion",
        "continent": "Continente"
    },
    title="PIB per Capita vs Esperanza de Vida — 2007 (tamaño = poblacion)",
)
fig.update_layout(
    legend_title="Continente",
    xaxis_title="PIB per Capita (USD, escala log)",
    yaxis_title="Esperanza de vida (años)"
)
fig.show()

In [ ]:
# Grafico de burbujas animado — visualizacion iconica de Hans Rosling (1952-2007)
# animation_frame: crea un slider y controles de reproduccion por año
# animation_group: vincula el mismo pais entre frames para animacion suave
fig = px.scatter(
    data,
    x="gdpPercap",
    y="lifeExp",
    color="continent",
    size="pop",
    size_max=60,
    hover_name="country",
    hover_data={"pop": True, "gdpPercap": True, "iso_alpha": False},
    log_x=True,
    animation_frame="year",
    animation_group="country",
    range_x=[200, 120000],
    range_y=[25, 90],
    labels={
        "gdpPercap": "PIB per capita (USD, escala log)",
        "lifeExp": "Esperanza de vida (años)",
        "pop": "Poblacion",
        "continent": "Continente",
        "year": "Año"
    },
    title="Evolucion del PIB per Capita vs Esperanza de Vida (1952–2007) — Hans Rosling",
)
fig.update_layout(
    legend_title="Continente",
    xaxis_title="PIB per Capita (USD, escala log)",
    yaxis_title="Esperanza de vida (años)"
)
fig.show()

## 5. Gráfico de Barras: Población Total por Continente (2007)

El **gráfico de barras** compara la magnitud de una variable entre categorías discretas. Las barras se ordenan de mayor a menor población para facilitar la lectura comparativa.

#### Datos clave
- **Asia** concentra más de la mitad de la población mundial (~4.000 M en 2007).
- **África** supera a Europa en habitantes, pese a estar frecuentemente infrarepresentada.
- **Oceanía** tiene la población más pequeña, comparable a un único país mediano.

> `texttemplate='%{text:.2s}'` usa notación científica abreviada (2s = 2 cifras significativas): `4.03G` = 4.030 millones.

In [ ]:
# Agrupar por continente y ordenar de mayor a menor poblacion
pop_by_continent = (
    data_2007
    .groupby('continent', as_index=False)
    .agg(Poblacion_Total=('pop', 'sum'))
    .sort_values('Poblacion_Total', ascending=False)
)

fig = px.bar(
    pop_by_continent,
    x="continent",
    y="Poblacion_Total",
    color="continent",
    text="Poblacion_Total",
    labels={"Poblacion_Total": "Poblacion Total (hab)", "continent": "Continente"},
    title="Poblacion Total por Continente (2007) — ordenado de mayor a menor",
)
fig.update_traces(texttemplate='%{text:.2s}', textposition='outside')
fig.update_layout(
    showlegend=False,
    xaxis_title="Continente",
    yaxis_title="Poblacion Total (hab)",
    xaxis={'categoryorder': 'total descending'}
)
fig.show()

## 6. Gráficos de Líneas: Evolución Temporal por Continente

El **gráfico de líneas** es el estándar para visualizar series temporales. Usamos la media por continente para mostrar la trayectoria de cada región, evitando la saturación de 142 líneas individuales.

### 6.1 Esperanza de Vida Media por Continente (1952–2007)

| Continente | Tendencia esperada |
|-----------|-------------------|
| Europa | Alta y estable; ligero incremento continuo |
| Oceanía | Similar a Europa, menos representativa (pocos países) |
| América | Crecimiento sostenido, convergencia con Europa |
| Asia | Ascenso espectacular: de ~46 años en 1952 a ~70 en 2007 |
| África | Mejora hasta los 80, estancamiento o retroceso en los 90 por el VIH/SIDA |

> `hovermode='x unified'` muestra los valores de todos los continentes simultáneamente al pasar el cursor sobre un año, facilitando la comparación.

In [ ]:
# Media de esperanza de vida por continente y año
mean_life = data.groupby(['continent', 'year'], as_index=False).agg({'lifeExp': 'mean'})

fig = px.line(
    mean_life,
    x="year",
    y="lifeExp",
    color="continent",
    markers=True,
    labels={
        "lifeExp": "Esperanza de vida media (años)",
        "year": "Año",
        "continent": "Continente"
    },
    title="Evolucion de la Esperanza de Vida Media por Continente (1952–2007)",
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(
    xaxis_title="Año",
    yaxis_title="Esperanza de vida media (años)",
    legend_title="Continente",
    hovermode="x unified"
)
fig.show()

### 6.2 PIB per Cápita Medio por Continente (1952–2007)

El PIB promedio por continente revela **brechas económicas históricas** y su evolución.

#### Puntos clave
- **Oceanía** lidera por su reducido número de países (Australia + Nueva Zelanda, ambos muy ricos).
- **Europa** muestra crecimiento sostenido, acelerado tras la reconstrucción post-WWII.
- **Asia** experimenta el crecimiento más rápido desde 1970, impulsado por los *Tigres Asiáticos* (Corea del Sur, Singapur, Hong Kong, Taiwán) y posteriormente China.
- La **brecha absoluta** entre continentes ricos y pobres se ha ampliado, aunque en términos relativos algunos han convergido.

In [ ]:
mean_gdp = data.groupby(['continent', 'year'], as_index=False).agg({'gdpPercap': 'mean'})

fig = px.line(
    mean_gdp,
    x="year",
    y="gdpPercap",
    color="continent",
    markers=True,
    labels={
        "gdpPercap": "PIB per capita medio (USD)",
        "year": "Año",
        "continent": "Continente"
    },
    title="Evolucion del PIB per Capita Medio por Continente (1952–2007)",
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(
    xaxis_title="Año",
    yaxis_title="PIB per Capita Medio (USD)",
    legend_title="Continente",
    hovermode="x unified"
)
fig.show()

### 6.3 Población Total por Continente (1952–2007)

La evolución demográfica refleja las **distintas fases de la transición demográfica** de cada región:

| Fase | Natalidad | Mortalidad | Resultado |
|------|-----------|------------|-----------|
| 1 — Pre-industrial | Alta | Alta | Población estable |
| 2 — Mejoras sanitarias | Alta | Baja | Crecimiento explosivo |
| 3 — Desarrollo | Baja | Baja | Crecimiento moderado |
| 4 — Sociedades maduras | Muy baja | Baja | Estabilidad o declive |

> **África** se encuentra entre las fases 2 y 3 (explica su crecimiento acelerado). **Europa** está en la fase 4 (natalidad muy baja, estabilización demográfica).

In [ ]:
# Suma total de poblacion por continente y año
total_pop = data.groupby(['continent', 'year'], as_index=False).agg({'pop': 'sum'})

fig = px.line(
    total_pop,
    x="year",
    y="pop",
    color="continent",
    markers=True,
    labels={
        "pop": "Poblacion Total (hab)",
        "year": "Año",
        "continent": "Continente"
    },
    title="Evolucion de la Poblacion Total por Continente (1952–2007)",
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(
    xaxis_title="Año",
    yaxis_title="Poblacion Total (hab)",
    legend_title="Continente",
    hovermode="x unified"
)
fig.show()

## 7. Histograma: Distribución de la Esperanza de Vida por Continente (2007)

El **histograma** agrupa los países en intervalos de esperanza de vida y muestra cuántos caen en cada rango. Al desglosar por continente con `barmode='overlay'`, se puede ver la contribución de cada región a la distribución global.

#### Conceptos estadísticos a observar

- **Bimodalidad**: Si aparecen dos picos separados, indica la existencia de dos grupos de países con perfiles muy distintos (p.ej. países desarrollados vs. en vías de desarrollo).
- **Dispersión intra-continental**: Una distribución ancha indica alta desigualdad dentro del grupo; estrecha indica homogeneidad.
- **Solapamiento**: Las áreas donde coinciden varios continentes muestran países con niveles similares de longevidad independientemente de su región.

In [ ]:
# Estadisticas globales de referencia
media_global = data_2007['lifeExp'].mean()
mediana_global = data_2007['lifeExp'].median()
print(f"Media global (2007)  : {media_global:.1f} años")
print(f"Mediana global (2007): {mediana_global:.1f} años")

fig = px.histogram(
    data_2007,
    x="lifeExp",
    color="continent",
    nbins=20,
    barmode="overlay",
    opacity=0.7,
    labels={"lifeExp": "Esperanza de vida (años)", "continent": "Continente"},
    title="Distribucion de la Esperanza de Vida por Continente (2007)",
)
fig.add_vline(
    x=mediana_global, line_dash="dash", line_color="black",
    annotation_text=f"Mediana: {mediana_global:.1f}", annotation_position="top left"
)
fig.update_layout(
    xaxis_title="Esperanza de vida (años)",
    yaxis_title="Numero de paises",
    legend_title="Continente",
    bargap=0.05
)
fig.show()

## 8. Diagramas de Caja y Bigotes

El **diagrama de caja** (boxplot) resume la distribución de una variable mediante 5 estadísticos clave, facilitando la comparación entre grupos.

| Elemento visual | Estadístico |
|----------------|-------------|
| Línea central | Mediana — percentil 50 |
| Borde inferior de la caja | Q1 — percentil 25 |
| Borde superior de la caja | Q3 — percentil 75 |
| Bigote inferior | Q1 − 1,5 · IQR |
| Bigote superior | Q3 + 1,5 · IQR |
| Puntos fuera de bigotes | **Outliers** — valores atípicos |

Con `points='all'` se superponen todos los países como puntos sobre la caja, revelando la distribución real en lugar de solo el resumen estadístico.

### 8.1 Esperanza de Vida por Continente (2007)

In [ ]:
continent_order = ['Africa', 'Americas', 'Asia', 'Europe', 'Oceania']

fig = px.box(
    data_2007,
    x="continent",
    y="lifeExp",
    color="continent",
    points="all",
    category_orders={"continent": continent_order},
    hover_name="country",
    labels={"lifeExp": "Esperanza de vida (años)", "continent": "Continente"},
    title="Distribucion de la Esperanza de Vida por Continente (2007)",
)
fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.update_layout(
    showlegend=False,
    xaxis_title="Continente",
    yaxis_title="Esperanza de vida (años)"
)
fig.show()

### 8.2 PIB per Cápita por Continente (2007)

El PIB per cápita tiene una distribución muy **sesgada a la derecha** dentro de cada continente: unos pocos países muy ricos elevan la media muy por encima de la mediana.

#### Outliers esperados por continente

| Continente | Outliers notables |
|-----------|------------------|
| Europa | Noruega, Luxemburgo, Suiza |
| Asia | Singapur, Kuwait, Japón |
| África | Guinea Ecuatorial, Gabón (petróleo) |
| América | EE.UU. y Canadá elevan la media; el resto de América queda por debajo |

> La asimetría (skewness) del PIB per cápita justifica el uso de **escala logarítmica** en el eje X del scatter plot de la sección 4.

In [ ]:
continent_order = ['Africa', 'Americas', 'Asia', 'Europe', 'Oceania']

fig = px.box(
    data_2007,
    x="continent",
    y="gdpPercap",
    color="continent",
    points="all",
    category_orders={"continent": continent_order},
    hover_name="country",
    labels={"gdpPercap": "PIB per capita (USD)", "continent": "Continente"},
    title="Distribucion del PIB per Capita por Continente (2007)",
)
fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.update_layout(
    showlegend=False,
    xaxis_title="Continente",
    yaxis_title="PIB per Capita (USD)"
)
fig.show()

## 9. Mapas de Calor

Un **mapa de calor** codifica una matriz de valores numéricos mediante una escala de color bidimensional. Es ideal para detectar patrones en tablas de correlaciones o en series temporales con múltiples categorías.

### 9.1 Correlaciones entre Variables (2007)

La **correlación de Pearson** mide la fuerza de la relación lineal entre dos variables:

| Valor | Interpretación |
|-------|----------------|
| 1.0 | Correlación positiva perfecta |
| 0.7–0.9 | Correlación positiva fuerte |
| 0.4–0.6 | Correlación positiva moderada |
| 0.0–0.3 | Correlación débil o nula |
| Negativo | Correlación inversa |

> La diagonal siempre vale 1 (una variable correlaciona perfectamente consigo misma). Solo se muestra el triángulo inferior para evitar redundancia.

#### Hipótesis previas
- `gdpPercap` ↔ `lifeExp`: correlación **positiva fuerte** — los países más ricos tienden a vivir más.
- `pop` ↔ otras variables: correlación **débil** — la población no determina el nivel de desarrollo.
- `gdpPercap` ↔ `pop`: posiblemente **nula o negativa** — los países más poblados (China, India) no son los más ricos per cápita.

In [ ]:
corr_data = data_2007[['lifeExp', 'pop', 'gdpPercap']].corr()

print("Matriz de correlacion de Pearson (2007):")
print(corr_data.round(3).to_string())
print("\nCorrelacion con esperanza de vida:")
print(corr_data['lifeExp'].drop('lifeExp').sort_values(ascending=False).round(3).to_string())

fig = px.imshow(
    corr_data,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    labels={"color": "Correlacion"},
    title="Mapa de Calor: Correlaciones entre Variables (2007)",
)
fig.update_layout(
    xaxis_title="",
    yaxis_title=""
)
fig.show()

### 9.2 Esperanza de Vida Media por Continente y Año (1952–2007)

Este mapa de calor combina la **dimensión geográfica** (continente, eje Y) y la **temporal** (año, eje X), usando el color para codificar la intensidad de la esperanza de vida media.

#### Ventajas sobre el gráfico de líneas

| Aspecto | Líneas | Calor |
|---------|--------|-------|
| Valores precisos | Sí | Aproximado (por color) |
| Comparación entre categorías | Moderada | Alta (vista simultánea) |
| Detección de cambios bruscos | Moderada | Alta (saltos de color) |

#### Qué buscar
- Transición progresiva de **colores fríos → cálidos** de izquierda a derecha en todas las filas → mejora global.
- **Fila de África**: más fría que el resto, especialmente a partir de 1987 (impacto VIH/SIDA).

In [ ]:
life_pivot = (
    data.groupby(['continent', 'year'], as_index=False)
    .agg({'lifeExp': 'mean'})
    .pivot(index='continent', columns='year', values='lifeExp')
)

fig = px.imshow(
    life_pivot,
    text_auto=".1f",
    aspect="auto",
    color_continuous_scale="RdYlGn",
    labels={"x": "Año", "y": "Continente", "color": "Esp. de vida media (años)"},
    title="Esperanza de Vida Media por Continente y Año (1952–2007)",
)
fig.update_layout(xaxis_title="Año", yaxis_title="Continente")
fig.show()

### 9.3 PIB per Cápita Medio por Continente y Año (1952–2007)

Mientras la esperanza de vida **converge** entre continentes, el PIB per cápita **diverge**: los continentes más ricos crecen más rápido en términos absolutos, ampliando la brecha económica global.

#### Lecturas clave
- **Oceanía y Europa**: las filas más brillantes, con crecimiento acelerado desde 1970.
- **Asia**: transición visible de colores oscuros a medios, especialmente desde 1980 — el "milagro económico" asiático.
- **África y Asia Meridional**: permanecen oscuras en toda la serie — el subdesarrollo económico persiste.

In [ ]:
gdp_pivot = (
    data.groupby(['continent', 'year'], as_index=False)
    .agg({'gdpPercap': 'mean'})
    .pivot(index='continent', columns='year', values='gdpPercap')
)

fig = px.imshow(
    gdp_pivot,
    text_auto=".0f",
    aspect="auto",
    color_continuous_scale="Viridis",
    labels={"x": "Año", "y": "Continente", "color": "PIB per capita medio (USD)"},
    title="PIB per Capita Medio por Continente y Año (1952–2007)",
)
fig.update_layout(xaxis_title="Año", yaxis_title="Continente")
fig.show()

### 9.4 Población Promedio por Continente y Año (1952–2007)

La población **media** por continente (en lugar de la suma) normaliza el efecto de tener diferente número de países por continente, mostrando el ritmo de crecimiento por país típico de cada región.

#### Patrón esperado
- **Asia**: domina en valor absoluto (China e India elevan la media continental).
- **África**: el continente con el crecimiento demográfico **más acelerado** en términos relativos.
- **Europa**: prácticamente estable — tasas de natalidad muy bajas desde los años 70.

In [ ]:
pop_pivot = (
    data.groupby(['continent', 'year'], as_index=False)
    .agg({'pop': 'mean'})
    .pivot(index='continent', columns='year', values='pop')
)

fig = px.imshow(
    pop_pivot,
    text_auto=".2s",
    aspect="auto",
    color_continuous_scale="Turbo",
    labels={"x": "Año", "y": "Continente", "color": "Poblacion media (hab)"},
    title="Poblacion Media por Continente y Año (1952–2007)",
)
fig.update_layout(xaxis_title="Año", yaxis_title="Continente")
fig.show()

## Conclusiones

### Hallazgos principales

1. **El mundo ha mejorado**: entre 1952 y 2007, la esperanza de vida global pasó de ~49 a ~67 años y el PIB per cápita se multiplicó aproximadamente por 3.
2. **La relación riqueza ↔ longevidad es fuerte pero no lineal**: a partir de ~10.000–15.000 USD/hab los retornos en años de vida disminuyen notablemente (rendimientos decrecientes).
3. **Asia es la gran historia de éxito**: Corea del Sur, Singapur y China pasaron de niveles de desarrollo bajos a altos en menos de dos generaciones.
4. **África sigue rezagada**: el VIH/SIDA estancó la esperanza de vida en los años 90, aunque hay señales de recuperación a partir de 2000.
5. **La población no determina el desarrollo**: India y China son los más poblados pero no los más ricos per cápita; Luxemburgo o Singapur, con muy poca población, lideran el PIB per cápita.

### Próximos pasos sugeridos
- Incorporar datos más recientes (2007–2023) desde la API de Gapminder.
- Analizar la desigualdad **dentro** de cada país (coeficiente de Gini).
- Construir un **modelo de regresión** para predecir la esperanza de vida a partir del PIB y otras variables.